<div style="text-align:center; line-height:1.6">

МИНИСТЕРСТВО НАУКИ И ВЫСШЕГО ОБРАЗОВАНИЯ РОССИЙСКОЙ ФЕДЕРАЦИИ<br>
Новосибирский государственный технический университет<br>
Факультет автоматики и вычислительной техники

# ОТЧЁТ
## по лабораторной работе № 4
### «MLOps»

Дисциплина «Методы анализа данных» · **Вариант 16** «Прогноз энергопотребности охлаждения здания»

</div>

<div style="text-align:right">
Выполнил: студент гр. АВТ-313 Савченко В.Э.<br>
Проверила: Ганелина Н.Д.
</div>

<div style="text-align:center">Новосибирск, 2026</div>

---

Блокнот повторяет ход работы: каждая ячейка — команда терминала (`!`) с её фактическим выводом. Исходники — в папке `src/`, скриншоты — в `screenshots/`.

# 1. Цель работы и постановка задачи

**Цель:** освоить основы MLOps — учёт и сравнение экспериментов в MLflow, регистрацию и управление версиями моделей, развёртывание сервиса предсказаний в Docker и Docker Compose, проверку его работоспособности, обновление и откат модели.

**Индивидуальный вариант 16 — прогноз энергопотребности охлаждения здания.**

| Пункт | Содержание варианта |
|---|---|
| Данные | UCI Energy Efficiency, https://archive.ics.uci.edu/dataset/242/energy+efficiency |
| Постановка | геометрия здания → Y2 (нагрузка охлаждения, Cooling Load) |
| Алгоритмы | Ridge и RandomForestRegressor |
| Критерий | MAE (дополнительно RMSE и R²) |
| Особенности | исключить Y1 из X; сравнить настройки; показать откат по версии API |

Задачи: подготовить данные и зафиксировать разбиение train/val/test; выполнить не менее шести экспериментов и записать в MLflow параметры, метрики, теги и артефакты; выбрать модель по validation MAE; зарегистрировать две версии в Model Registry и назначить alias `champion`; экспортировать модель вместе с preprocessing; реализовать API на FastAPI; собрать Docker-образ и запустить контейнер; проверить корректные и некорректные запросы и совпадение ответов API с экспортированной моделью; показать обновление и откат без переобучения; запустить сервис через Docker Compose с healthcheck; зафиксировать зависимости, данные и разбиение.

# 2. Набор данных

**Источник:** Tsanas A., Xifara A. Energy Efficiency, UCI Machine Learning Repository, id 242 (файл `ENB2012_data`). Смоделировано 768 зданий (12 форм × 4 ориентации × вариантов остекления) в Ecotect. Сохранён в проекте как `data/ENB2012_data.csv`, SHA-256 = `46fc2a38d43879311f0bd09441dbcb02594bc41f11a382c452f095fa1d752c18`.

| Столбец | Имя в проекте | Описание | Диапазон |
|---|---|---|---|
| X1 | relative_compactness | относительная компактность | 0,62–0,98 |
| X2 | surface_area | площадь поверхности, м² | 514,5–808,5 |
| X3 | wall_area | площадь стен, м² | 245–416,5 |
| X4 | roof_area | площадь крыши, м² | 110,25–220,5 |
| X5 | overall_height | высота, м | 3,5 / 7 |
| X6 | orientation | ориентация (код 2–5) | категориальный |
| X7 | glazing_area | доля остекления | 0–0,4 |
| X8 | glazing_area_distribution | распределение остекления (код 0–5) | категориальный |
| Y1 | heating_load | нагрузка отопления — **исключена из X** | 6,0–43,1 |
| **Y2** | **cooling_load** | **нагрузка охлаждения, кВт·ч/м² — цель** | 10,9–48,0 (среднее 24,6) |

**Подготовка.** Пропусков и дубликатов нет. Y1 исключена из признаков: это вторая целевая переменная того же расчёта, в момент прогноза она неизвестна, а корреляция Y1–Y2 = 0,976 дала бы утечку. В коде это проверяет `assert`, а API отвечает 422 на поле `heating_load`. Все признаки приведены к `float64` (единый тип входа). Preprocessing входит в `Pipeline` и обучается только на train: `StandardScaler` для 6 числовых признаков, `OneHotEncoder(handle_unknown="ignore")` для кодовых `orientation` и `glazing_area_distribution`.

**Разбиение** (фиксировано, `random_state = 42`, хранится в `data/split.json` с индексами строк и хэшем данных): train 60 % (460), validation 20 % (154) — выбор модели, test 20 % (154) — только итоговая оценка.

In [1]:
import sys, mlflow, sklearn, pandas as pd
sys.path.insert(0, "src"); import config as C
print(sys.version.split()[0], mlflow.__version__, sklearn.__version__)
df = C.load_data(); print(df.shape); print("пропуски:", df.isna().sum().sum(), "| дубликаты:", df.duplicated().sum())
print(df.describe().T[["mean","std","min","max"]].round(2).to_string())
print("corr(Y1, Y2) =", round(df.heating_load.corr(df.cooling_load), 3))
s = C.get_split(df); print(s["scheme"], s["sizes"], s["data_sha256"][:16])

3.12.13 3.5.0 1.7.2
(768, 10)
пропуски: 0 | дубликаты: 0
                             mean    std     min     max
relative_compactness         0.76   0.11    0.62    0.98
surface_area               671.71  88.09  514.50  808.50
wall_area                  318.50  43.63  245.00  416.50
roof_area                  176.60  45.17  110.25  220.50
overall_height               5.25   1.75    3.50    7.00
orientation                  3.50   1.12    2.00    5.00
glazing_area                 0.23   0.13    0.00    0.40
glazing_area_distribution    2.81   1.55    0.00    5.00
heating_load                22.31  10.09    6.01   43.10
cooling_load                24.59   9.51   10.90   48.03
corr(Y1, Y2) = 0.976
train 60% / val 20% / test 20% {'train': 460, 'val': 154, 'test': 154} 46fc2a38d4387931


# 3. Программное окружение и структура проекта

ОС Ubuntu 24.04, Python 3.12.13, Docker Engine 29.9.2, Docker Compose v5.7.0. Прямые зависимости закреплены в `requirements.txt`, полный снимок (133 пакета) — в `requirements.lock.txt`; для образа API — облегчённый `requirements-api.txt` (`mlflow-skinny` вместо полного MLflow: серверу и UI в контейнере делать нечего).

| Пакет | Версия | | Пакет | Версия |
|---|---|---|---|---|
| mlflow | 3.5.0 | | fastapi | 0.118.0 |
| scikit-learn | 1.7.2 | | uvicorn | 0.37.0 |
| pandas | 2.2.3 | | pydantic | 2.11.9 |
| numpy | 2.1.3 | | sqlalchemy | 2.0.43 |

```
mlops_lab/
├── data/ENB2012_data.csv, split.json      # снимок данных и фиксированное разбиение
├── src/config.py      # пути, имена, признаки, загрузка, разбиение
├── src/train.py       # обучение и run в MLflow (--register — регистрация)
├── src/promote.py     # alias champion -> версия
├── src/export_model.py# выгрузка champion в exported_model + эталоны
├── src/evaluate.py    # итоговая оценка на test
├── src/app.py         # FastAPI: /health, /metadata, /predict
├── src/check_api.py   # проверка совпадения ответов и ошибок
├── requirements.txt, requirements.lock.txt, requirements-api.txt
├── Dockerfile, .dockerignore, compose.yaml
├── exported_model/    # MLmodel, model.pkl, окружение, metadata.json
├── sample_request.json, expected_predictions.json, test_metrics.json
└── mlflow-data/       # SQLite + artifacts сервера MLflow
```

Запуск сервера MLflow (метаданные в SQLite, артефакты — локальная папка):

```bash
python3.12 -m venv .venv && source .venv/bin/activate
python -m pip install -r requirements.txt
mlflow server --backend-store-uri sqlite:///mlflow-data/mlflow.db \
  --artifacts-destination mlflow-data/artifacts --host 127.0.0.1 --port 5000
```

# 4. Эксперименты в MLflow

`train.py` загружает данные, читает `split.json`, строит Pipeline, обучает его на train и считает MAE, RMSE, R² на train и val. В run записываются: **параметры** (algorithm, alpha / trees / depth, seed, размеры выборок), **метрики** (train_* и val_*), **теги** (variant = 16, target, excluded_from_X = Y1, data_sha256, split, purpose), **датасет** (`log_input`), **артефакты** — `split.json`, исходный код, `requirements.txt`, график «факт–прогноз» на val и сама модель с signature и input_example.


**Таблица 1 — Сравнение шести экспериментов (сортировка по val MAE)**

| № | run_id | Алгоритм | Гиперпараметры | train MAE | val MAE | val RMSE | val R² |
|---|---|---|---|---|---|---|---|
| 6 | 48947fcbeb41… | RandomForest | trees = 200, depth = 12 | 0,410 | **1,090** | **1,819** | **0,962** |
| 5 | 1e1a488950a0… | RandomForest | trees = 100, depth = 8 | 0,599 | 1,116 | 1,868 | 0,960 |
| 4 | 4988b8949188… | RandomForest | trees = 50, depth = 4 | 1,284 | 1,552 | 2,284 | 0,940 |
| 1 | 2dc5e69ee005… | Ridge | alpha = 0,1 | 2,255 | 2,021 | 2,738 | 0,914 |
| 2 | 3558fe77f066… | Ridge | alpha = 1 | 2,260 | 2,033 | 2,750 | 0,913 |
| 3 | 40b2070d7a5b… | Ridge | alpha = 10 | 2,315 | 2,142 | 2,851 | 0,907 |

**Анализ настроек.** У Ridge рост alpha монотонно ухудшает val MAE (2,02 → 2,14): модель недообучена, регуляризация лишь сильнее «прижимает» коэффициенты. Линейная модель упирается в R² ≈ 0,91: зависимость Y2 от геометрии нелинейна (резкий скачок между зданиями высотой 3,5 и 7 м, взаимодействие компактности и площадей). Случайный лес улучшается с ростом глубины и числа деревьев: при depth = 4 он слишком прост (MAE 1,55), при depth = 8 и 12 MAE ≈ 1,1. Разница между depth = 8 и 12 мала (0,026), разрыв train/val у леса больше (0,41 / 1,09), но val-ошибка не растёт — переобучения, ухудшающего обобщение, нет.

In [2]:
!python src/train.py --algorithm ridge --alpha 0.1

run_id=2dc5e69ee00543ee9fc3a3ed022fc131
model_uri=models:/m-b884ff5cd9eb4b0eb4f69536f7991559
val: val_mae=2.0212, val_rmse=2.7376, val_r2=0.9141


In [3]:
!python src/train.py --algorithm ridge --alpha 1

run_id=3558fe77f06647f4af09e0f598840fbd
model_uri=models:/m-5ab6712130d441b1a478a3dc20052b1b
val: val_mae=2.0330, val_rmse=2.7502, val_r2=0.9133


In [4]:
!python src/train.py --algorithm ridge --alpha 10

run_id=40b2070d7a5b407c816fa84a2c677ecf
model_uri=models:/m-f62dc1eec2f643cdbadcacb1b60ecc4f
val: val_mae=2.1415, val_rmse=2.8510, val_r2=0.9068


In [5]:
!python src/train.py --algorithm rf --trees 50 --depth 4

run_id=4988b89491884a53960cf5f5a2314c22
model_uri=models:/m-4eae462924fe4f158b14114fdb24fa68
val: val_mae=1.5517, val_rmse=2.2837, val_r2=0.9402


In [6]:
!python src/train.py --algorithm rf --trees 100 --depth 8

run_id=1e1a488950a04f57adecf1f6ae55748e
model_uri=models:/m-95b0a858d0de4a48bfc2c7c276baa1d8
val: val_mae=1.1162, val_rmse=1.8684, val_r2=0.9600


In [7]:
!python src/train.py --algorithm rf --trees 200 --depth 12

run_id=48947fcbeb414c488f1daa052aa0d242
model_uri=models:/m-b858b185378b48c2a417ff23802dee19
val: val_mae=1.0898, val_rmse=1.8187, val_r2=0.9621


![Рисунок 1 — Выполнение шести экспериментов: run_id, model_uri и validation-метрики](screenshots/term_train6.png)

*Рисунок 1 — Выполнение шести экспериментов: run_id, model_uri и validation-метрики*

![Рисунок 2 — Запуски в эксперименте lab4-energy-cooling в интерфейсе MLflow](screenshots/mlflow_runs.png)

*Рисунок 2 — Запуски в эксперименте lab4-energy-cooling в интерфейсе MLflow*

![Рисунок 3 — Сравнение шести запусков: параметры и метрики](screenshots/mlflow_compare.png)

*Рисунок 3 — Сравнение шести запусков: параметры и метрики*

![Рисунок 4 — Страница запуска rf-trees200-depth12: параметры, метрики, теги](screenshots/mlflow_run.png)

*Рисунок 4 — Страница запуска rf-trees200-depth12: параметры, метрики, теги*

![Рисунок 5 — Артефакты запуска: модель, код, окружение, разбиение, график](screenshots/mlflow_artifacts.png)

*Рисунок 5 — Артефакты запуска: модель, код, окружение, разбиение, график*

# 5. Выбор модели и итоговая оценка

**Правило выбора (задано до просмотра test):** минимальный validation MAE; при разнице < 0,01 — более простая модель. Победитель — **RandomForestRegressor (200 деревьев, max_depth = 12)**: val MAE = 1,090 кВт·ч/м², почти вдвое меньше лучшего Ridge (2,021). Ridge (alpha = 0,1) оставлен второй версией как простая и интерпретируемая альтернатива и как кандидат для демонстрации обновления.

Итоговая оценка экспортированной модели на отложенном test (154 здания, выполнена один раз, гиперпараметры после неё не менялись):

| Метрика | test | для сравнения: прогноз средним train |
|---|---|---|
| MAE, кВт·ч/м² | **1,173** | 8,610 |
| RMSE | 1,788 | — |
| R² | 0,966 | 0 |

Test MAE (1,17) близок к val MAE (1,09) — оценка на validation не была оптимистичной. Средняя ошибка ≈ 4,8 % от среднего Y2 (24,6) и в 7,3 раза меньше, чем у наивного прогноза средним.

# 6. Регистрация версий и alias champion

Выбранные конфигурации обучены повторно с флагом `--register` (два новых run, всего в эксперименте 8). Каждый вызов создаёт новую версию модели `cooling-load-regressor`:


| Версия | Алгоритм | run_id | val MAE | Alias |
|---|---|---|---|---|
| 1 | RandomForest, 200 / 12 | 2dd23df047f34100b250c973f5342bb9 | 1,090 | **champion** |
| 2 | Ridge, alpha = 0,1 | 32c8d50a7670466f8e896522a1739506 | 2,021 | — |

Alias — это указатель на версию: `promote.py` меняет его без создания новой модели. `export_model.py` один раз разрешает `champion` в конкретный номер, скачивает эту версию в `exported_model/` (MLmodel, `model.pkl` — весь Pipeline с preprocessing, `requirements.txt`, `python_env.yaml`) и пишет `metadata.json` с номером версии, run_id, параметрами и порядком признаков. Там же формируются `sample_request.json` (3 реальных test-объекта) и `expected_predictions.json` (ответы экспортированной модели).

In [8]:
!python src/train.py --algorithm rf --trees 200 --depth 12 --register

run_id=2dd23df047f34100b250c973f5342bb9
model_uri=models:/m-a93a28681ae1470cb2e35b7a450c5905
val: val_mae=1.0898, val_rmse=1.8187, val_r2=0.9621
REGISTERED_VERSION=1


In [9]:
!python src/train.py --algorithm ridge --alpha 0.1 --register

run_id=32c8d50a7670466f8e896522a1739506
model_uri=models:/m-91428131c47b4a8cb3dec16d1934d548
val: val_mae=2.0212, val_rmse=2.7376, val_r2=0.9141
REGISTERED_VERSION=2


In [10]:
!python src/promote.py --version 1

cooling-load-regressor@champion -> version 1 (run_id=2dd23df047f34100b250c973f5342bb9)


In [11]:
!python src/export_model.py

exported cooling-load-regressor version 1 (rf) -> /home/claude/mlops_lab/exported_model
expected predictions: [25.1522, 23.5081, 11.3752]


In [12]:
!python src/evaluate.py

{
  "model_version": "1",
  "algorithm": "rf",
  "n_test": 154,
  "test_mae": 1.1731,
  "test_rmse": 1.7876,
  "test_r2": 0.9657,
  "baseline_mean_mae": 8.6101
}


In [13]:
!ls exported_model

MLmodel
conda.yaml
input_example.json
metadata.json
model.pkl
python_env.yaml
registered_model_meta
requirements.txt
serving_input_example.json


In [14]:
!cat exported_model/metadata.json

{
  "model_name": "cooling-load-regressor",
  "model_version": "1",
  "alias_at_export": "champion",
  "run_id": "2dd23df047f34100b250c973f5342bb9",
  "params": {
    "algorithm": "rf",
    "trees": "200",
    "depth": "12",
    "seed": "42",
    "n_train": "460",
    "n_val": "154",
    "n_features": "8"
  },
  "val_mae": 1.0897962658472478,
  "val_rmse": 1.818668252810098,
  "features": [
    "relative_compactness",
    "surface_area",
    "wall_area",
    "roof_area",
    "overall_height",
    "orientation",
    "glazing_area",
    "glazing_area_distribution"
  ],
  "target": "cooling_load (Y2), кВт·ч/м²",
  "excluded": "heating_load (Y1)",
  "data_sha256": "46fc2a38d43879311f0bd09441dbcb02594bc41f11a382c452f095fa1d752c18",
  "exported_at": "2026-11-10T12:25:26+00:00"
}


In [15]:
!cat sample_request.json

{
  "instances": [
    {
      "relative_compactness": 0.9,
      "surface_area": 563.5,
      "wall_area": 318.5,
      "roof_area": 122.5,
      "overall_height": 7.0,
      "orientation": 5.0,
      "glazing_area": 0.0,
      "glazing_area_distribution": 0.0
    },
    {
      "relative_compactness": 0.86,
      "surface_area": 588.0,
      "wall_area": 294.0,
      "roof_area": 147.0,
      "overall_height": 7.0,
      "orientation": 4.0,
      "glazing_area": 0.0,
      "glazing_area_distribution": 0.0
    },
    {
      "relative_compactness": 0.71,
      "surface_area": 710.5,
      "wall_area": 269.5,
      "roof_area": 220.5,
      "overall_height": 3.5,
      "orientation": 3.0,
      "glazing_area": 0.0,
      "glazing_area_distribution": 0.0
    }
  ]
}


![Рисунок 6 — Две версии cooling-load-regressor в Model Registry, версия 1 — champion](screenshots/mlflow_registry.png)

*Рисунок 6 — Две версии cooling-load-regressor в Model Registry, версия 1 — champion*

![Рисунок 7 — Регистрация, назначение champion, экспорт, test-метрики и metadata.json](screenshots/term_register_export.png)

*Рисунок 7 — Регистрация, назначение champion, экспорт, test-метрики и metadata.json*

# 7. API сервиса предсказаний

FastAPI-приложение `src/app.py` загружает модель один раз при старте (lifespan) из `MODEL_DIR` и не обращается к MLflow-серверу.

| Метод | Путь | Назначение |
|---|---|---|
| GET | /health | `{"status":"ok","model_version":"1"}` — приложение запущено, модель загружена |
| GET | /metadata | содержимое metadata.json: версия, run_id, параметры, порядок признаков |
| POST | /predict | прогноз Y2 для пакета зданий |
| GET | /docs | Swagger UI |

**Формат входа:** `{"instances": [ {8 признаков}, … ]}` — от 1 до 1000 объектов, каждый объект содержит ровно 8 числовых полей (`relative_compactness`, `surface_area`, `wall_area`, `roof_area`, `overall_height`, `orientation`, `glazing_area`, `glazing_area_distribution`). Pydantic проверяет наличие и тип полей, запрещает лишние поля (`extra="forbid"`) — в том числе Y1, — запрещает NaN/inf и контролирует допустимые диапазоны (площади > 0, остекление 0–1). **Формат ответа:** `predictions` — список float (регрессия, без округления к int), `unit`, `model_name`, `model_version`, `latency_ms`.

**Таблица 2 — Проверка запросов**

| Запрос | Ожидание | Результат |
|---|---|---|
| sample_request.json (3 объекта) | 200 | 200, [25,152; 23,508; 11,375] |
| один корректный объект | 200 | 200, [25,152] |
| только relative_compactness (нет 7 признаков) | 422 | 422, type = missing |
| пустой пакет `{"instances": []}` | 422 | 422, type = too_short |
| surface_area = "big" (неверный тип) | 422 | 422, type = float_parsing |
| лишнее поле heating_load (Y1) | 422 | 422, type = extra_forbidden |

Ни один ошибочный запрос не привёл к 500 — валидация входа контролируемая.

# 8. Docker-образ и контейнер

Dockerfile: `FROM python:3.12-slim`, рабочая папка `/app`, установка `requirements-api.txt`, копирование `src/app.py` и `exported_model/`, запуск от непривилегированного пользователя `USER 10001`, `EXPOSE 8000`, uvicorn слушает `0.0.0.0:8000` (иначе опубликованный порт недоступен снаружи контейнера); на хосте порт опубликован только на `127.0.0.1`. `.dockerignore` исключает `.venv`, `mlflow-data`, данные и скриншоты, но не `exported_model`.


**Совпадение API и экспортированной модели:** `check_api.py` сравнивает ответ `/predict` на `sample_request.json` с `expected_predictions.json` (`np.allclose`, atol = 1e-9) и номер версии из `/metadata`: API [25,1522; 23,5081; 11,3752] = экспорт [25,1522; 23,5081; 11,3752], **MATCH: True**. Это проверка доставки (та же модель и тот же preprocessing), а не качества — качество оценено на test в разделе 5.

В логах старта есть два предупреждения MLflow: несовпадение `pyarrow` (нужен только для логирования датасетов при обучении, на предсказание не влияет) и отказ записи трассировки в `/app/mlruns` (контейнер работает от непривилегированного пользователя без права записи в `/app` — это ожидаемо).

In [16]:
!docker build -t cooling-api:v1 .

#1 [internal] load build definition from Dockerfile
#1 DONE 0.0s
#2 [internal] load metadata for docker.io/library/python:3.12-slim
#2 DONE 0.0s
#3 [internal] load .dockerignore
#3 DONE 0.0s
#4 [1/6] FROM docker.io/library/python:3.12-slim@sha256:c23cb8f2a64ad0b8e2c88b78bb90132c815cfe141508ad6c738bbf628723c233
#5 [internal] load build context
#5 DONE 0.1s
#4 [1/6] FROM docker.io/library/python:3.12-slim@sha256:c23cb8f2a64ad0b8e2c88b78bb90132c815cfe141508ad6c738bbf628723c233
#4 DONE 3.3s
#6 [2/6] WORKDIR /app
#6 DONE 0.6s
#7 [3/6] COPY requirements-api.txt .
#7 DONE 0.0s
#8 [4/6] RUN python -m pip install --no-cache-dir -r requirements-api.txt
#8 0.430 Requirement already satisfied: mlflow-skinny==3.5.0 in /usr/local/lib/python3.12/site-packages (from -r requirements-api.txt (line 1)) (3.5.0)
#8 0.430 Requirement already satisfied: scikit-learn==1.7.2 in /usr/local/lib/python3.12/site-packages (from -r requirements-api.txt (line 2)) (1.7.2)
#8 0.430 Requirement already satisfied: pandas

In [17]:
!docker image ls cooling-api

IMAGE            ID             DISK USAGE   CONTENT SIZE   EXTRA
cooling-api:v1   dc6febee877a        592MB          131MB


In [18]:
!docker image inspect cooling-api:v1 --format '{{.Id}}'

sha256:dc6febee877a66acce5daf8ec8a15c3d2111ebdb8ff5322c4c0a1f91f29804aa


In [19]:
!docker run -d --name cooling-api -p 127.0.0.1:8000:8000 cooling-api:v1

9eb9e6b2d1b8f42d3b04e0385d62f63cd326d13e8fa471de2f06a014f2bdf2e8


In [20]:
!docker ps

CONTAINER ID   IMAGE            STATUS         PORTS                      NAMES
9eb9e6b2d1b8   cooling-api:v1   Up 8 seconds   127.0.0.1:8000->8000/tcp   cooling-api


In [21]:
!docker logs cooling-api

INFO:     Started server process [1]
INFO:     Waiting for application startup.
2026/11/10 12:27:30 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - pyarrow (current: uninstalled, required: pyarrow==21.0.0)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.
2026/11/10 12:27:32 WARNING mlflow.tracing.provider: An error occurred while disabling or re-enabling tracing: [Errno 13] Permission denied: '/app/mlruns' The original function will still be executed, but the tracing state may not be as expected. For full traceback, set logging level to debug.
INFO:     model loaded: cooling-load-regressor v1
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


In [22]:
!curl -f http://127.0.0.1:8000/health

{"status":"ok","model_version":"1"}


In [23]:
!curl -f -X POST http://127.0.0.1:8000/predict -H "Content-Type: application/json" --data-binary @sample_request.json

{"predictions":[25.15220000000004,23.508124999999982,11.375150000000026],"unit":"kWh/m2","model_name":"cooling-load-regressor","model_version":"1","latency_ms":41.3}


In [24]:
!python src/check_api.py

API model_version: 1 | expected: 1
API predictions:      [25.1522, 23.5081, 11.3752]
Exported predictions: [25.1522, 23.5081, 11.3752]
MATCH: True
single object (200)        -> 200 OK
missing features (422)     -> 422 OK
empty batch (422)          -> 422 OK
extra field Y1 (422)       -> 422 OK
wrong type (422)           -> 422 OK
exit code: 0


In [25]:
!curl -i -X POST http://127.0.0.1:8000/predict -H 'Content-Type: application/json' -d '{"instances": [{"relative_compactness": 0.9}]}'

HTTP/1.1 422 Unprocessable Entity
{"detail":[{"type":"missing","loc":["body","instances",0,"surface_area"],"msg":"Field required","input":{"relative_compactness":0.9}},{"type":"missing","loc":["body","instances",0,"wall_area"],"msg":"Field required","input":{"relative_compactness":0.9}},{"type":"missing","loc":["body","instances",0,"roof_area"],"msg":"Field requ


In [26]:
!curl -i -X POST http://127.0.0.1:8000/predict -H 'Content-Type: application/json' -d '{"instances": []}'

HTTP/1.1 422 Unprocessable Entity
{"detail":[{"type":"too_short","loc":["body","instances"],"msg":"List should have at least 1 item after validation, not 0","input":[],"ctx":{"field_type":"List","min_length":1,"actual_length":0}}]}


In [27]:
!curl -i ... (surface_area: "big" — неверный тип)

HTTP/1.1 422 Unprocessable Entity
{"detail":[{"type":"float_parsing","loc":["body","instances",0,"surface_area"],"msg":"Input should be a valid number, unable to parse string as a number","input":"big"}]}


In [28]:
!curl -i ... (лишнее поле heating_load = Y1)

HTTP/1.1 422 Unprocessable Entity
{"detail":[{"type":"extra_forbidden","loc":["body","instances",0,"heating_load"],"msg":"Extra inputs are not permitted","input":15.5}]}


In [29]:
!curl -i ... (один корректный объект)

HTTP/1.1 200 OK
{"predictions":[25.15220000000004],"unit":"kWh/m2","model_name":"cooling-load-regressor","model_version":"1","latency_ms":40.17}


![Рисунок 8 — Сборка образа cooling-api:v1, запуск контейнера, запросы и check_api.py](screenshots/term_docker_v1.png)

*Рисунок 8 — Сборка образа cooling-api:v1, запуск контейнера, запросы и check_api.py*

![Рисунок 9 — Swagger UI: POST /predict, код 200 и прогнозы модели версии 1](screenshots/swagger_predict.png)

*Рисунок 9 — Swagger UI: POST /predict, код 200 и прогнозы модели версии 1*

![Рисунок 10 — Ответы 422 на неполные признаки, пустой пакет, неверный тип и лишнее поле; 200 на один объект](screenshots/term_errors.png)

*Рисунок 10 — Ответы 422 на неполные признаки, пустой пакет, неверный тип и лишнее поле; 200 на один объект*

# 9. Обновление и откат

Демонстрация доставки (вариант требует «показать откат по версии API»): на время в сервис выпускается версия 2 (Ridge), затем выполняется откат к сохранённому образу с версией 1. Выпуск v2 — упражнение в доставке, а не признание её лучшей.


| Этап | Docker-образ | Registry-версия | Алгоритм | /metadata model_version | Ответы API на sample |
|---|---|---|---|---|---|
| исходно | cooling-api:v1 (dc6febee877a) | 1 (champion) | RF | 1 | 25,152 / 23,508 / 11,375 |
| обновление | cooling-api:v2 (28924e1c4492) | 2 (champion) | Ridge | 2 | 26,986 / 26,358 / 9,707 |
| откат | cooling-api:v1 (тот же образ) | 1 (champion) | RF | 1 | 25,152 / 23,508 / 11,375 |

Откат занял две команды Docker: образ v1 хранится локально, поэтому ни переобучение, ни пересборка не нужны. Тег образа (v1/v2) и номер модели — разные сущности; фактическая версия модели берётся из `metadata.json` внутри образа и видна в `/metadata` и в каждом ответе `/predict`.

In [30]:
!curl -s http://127.0.0.1:8000/metadata   # до обновления

{'model_name': 'cooling-load-regressor', 'model_version': '1', 'run_id': '2dd23df047f34100b250c973f5342bb9'} rf


In [31]:
!python src/promote.py --version 2

cooling-load-regressor@champion -> version 2 (run_id=32c8d50a7670466f8e896522a1739506)


In [32]:
!python src/export_model.py

exported cooling-load-regressor version 2 (ridge) -> /home/claude/mlops_lab/exported_model
expected predictions: [26.9864, 26.3582, 9.7068]


In [33]:
!docker build -t cooling-api:v2 .

sha256:28924e1c44925e0aae8d2a24af197e866d2a756c6e4d2f9fad4e0b2f20778d4f


In [34]:
!docker stop cooling-api && docker rm cooling-api

cooling-api


In [35]:
!docker run -d --name cooling-api -p 127.0.0.1:8000:8000 cooling-api:v2

d3d8ba31665fa7b25a6ce551905a4a725090d6ed8945bf2fd480f486afd4e124


In [36]:
!curl -f http://127.0.0.1:8000/metadata

{'model_name': 'cooling-load-regressor', 'model_version': '2', 'run_id': '32c8d50a7670466f8e896522a1739506'} ridge


In [37]:
!python src/check_api.py | head -4

API model_version: 2 | expected: 2
API predictions:      [26.9864, 26.3582, 9.7068]
Exported predictions: [26.9864, 26.3582, 9.7068]
MATCH: True

===== ОТКАТ =====


In [38]:
!docker stop cooling-api && docker rm cooling-api

cooling-api


In [39]:
!docker run -d --name cooling-api -p 127.0.0.1:8000:8000 cooling-api:v1   # без пересборки

5d8ea9ea18b5bdf6f93f5c8264b4dc5d478c0c2b96ca127f7a8b8aab46f3030a


In [40]:
!python src/promote.py --version 1

cooling-load-regressor@champion -> version 1 (run_id=2dd23df047f34100b250c973f5342bb9)


In [41]:
!curl -f http://127.0.0.1:8000/metadata

{'model_name': 'cooling-load-regressor', 'model_version': '1', 'run_id': '2dd23df047f34100b250c973f5342bb9'} rf


In [42]:
!python src/export_model.py   # выравниваем эталон с v1

exported cooling-load-regressor version 1 (rf) -> /home/claude/mlops_lab/exported_model
expected predictions: [25.1522, 23.5081, 11.3752]


In [43]:
!python src/check_api.py | head -4

API model_version: 1 | expected: 1
API predictions:      [25.1522, 23.5081, 11.3752]
Exported predictions: [25.1522, 23.5081, 11.3752]
MATCH: True


In [44]:
!docker image ls cooling-api

IMAGE            ID             DISK USAGE   CONTENT SIZE   EXTRA
cooling-api:v1   dc6febee877a        592MB          131MB   U    
cooling-api:v2   28924e1c4492        582MB          130MB


In [45]:
!docker inspect cooling-api --format '{{.Config.Image}}'

cooling-api:v1


![Рисунок 11 — Обновление до версии 2 и откат к образу v1](screenshots/term_update_rollback.png)

*Рисунок 11 — Обновление до версии 2 и откат к образу v1*

# 10. Docker Compose и healthcheck

`compose.yaml` описывает сервис `api`: сборка из текущего каталога, образ `cooling-api:${IMAGE_TAG:-v1}`, порт `127.0.0.1:8000:8000`, `restart: unless-stopped` и healthcheck — Python-запрос к `/health` внутри контейнера каждые 10 с (timeout 5 с, 3 попытки, start_period 20 с). Перед `up --build` champion возвращён на версию 1, поэтому тег v1 собирается из той же модели. Volume не нужен — модель встроена в образ; сервер MLflow остаётся на хосте.


Результат: статус контейнера **healthy**, FailingStreak = 0, все проверки с ExitCode 0, `/health` → `{"status":"ok","model_version":"1"}`.

**Эксплуатационная проверка.** Пять одинаковых запросов (3 объекта): latency_ms = 46,1; 40,5; 45,2; 26,4; 43,2 мс (время построения таблицы и предсказания в обработчике, без сети). Контейнер потребляет ≈ 166 МиБ памяти и 0,2 % CPU в простое. В логи пишутся версия, число строк и латентность, но не содержимое запроса. Healthcheck подтверждает работу процесса и загрузку модели, но не качество прогнозов и не дрейф данных — для этого нужны новые размеченные данные.

In [46]:
!docker stop cooling-api && docker rm cooling-api

cooling-api


In [47]:
!docker compose up -d --build

 Network mlops_lab_default Created 
 Network mlops_lab_default Created 
 Container cooling-api-compose Creating 
 Container cooling-api-compose Created 
 Container cooling-api-compose Starting 
 Container cooling-api-compose Started


In [48]:
!docker compose ps

NAME                  IMAGE            COMMAND                  SERVICE   CREATED          STATUS                    PORTS
cooling-api-compose   cooling-api:v1   "python -m uvicorn a…"   api       30 seconds ago   Up 30 seconds (healthy)   127.0.0.1:8000->8000/tcp


In [49]:
!docker compose logs --tail 6

cooling-api-compose  | INFO:     model loaded: cooling-load-regressor v1
cooling-api-compose  | INFO:     Application startup complete.
cooling-api-compose  | INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
cooling-api-compose  | INFO:     127.0.0.1:46434 - "GET /health HTTP/1.1" 200 OK
cooling-api-compose  | INFO:     127.0.0.1:44802 - "GET /health HTTP/1.1" 200 OK
cooling-api-compose  | INFO:     127.0.0.1:51936 - "GET /health HTTP/1.1" 200 OK


In [50]:
!curl -f http://127.0.0.1:8000/health

{"status":"ok","model_version":"1"}


In [51]:
!docker inspect cooling-api-compose --format '{{json .State.Health}}'

Status: healthy | FailingStreak: 0
  2026-11-10T12:28:44 ExitCode 0
  2026-11-10T12:28:54 ExitCode 0
  2026-11-10T12:29:04 ExitCode 0


In [52]:
!docker compose stats --no-stream

NAME                  CPU %     MEM USAGE / LIMIT
cooling-api-compose   0.18%     165MiB / 7.835GiB


In [53]:
!for i in 1 2 3 4 5; do curl -s -X POST http://127.0.0.1:8000/predict -H 'Content-Type: application/json' --data-binary @sample_request.json | python -c "import json,sys;print(json.load(sys.stdin)['latency_ms'])"; done

46.12
40.54
45.15
26.42
43.18


In [54]:
!docker logs --tail 8 cooling-api

INFO:     predict version=1 rows=3 latency_ms=40.54
INFO:     172.17.0.1:55526 - "POST /predict HTTP/1.1" 200 OK
INFO:     predict version=1 rows=3 latency_ms=45.15
INFO:     172.17.0.1:55532 - "POST /predict HTTP/1.1" 200 OK
INFO:     predict version=1 rows=3 latency_ms=26.42
INFO:     172.17.0.1:55534 - "POST /predict HTTP/1.1" 200 OK
INFO:     predict version=1 rows=3 latency_ms=43.18
INFO:     172.17.0.1:55544 - "POST /predict HTTP/1.1" 200 OK


In [55]:
!docker stats --no-stream cooling-api

NAME          CPU %     MEM USAGE / LIMIT     PIDS
cooling-api   0.19%     165.8MiB / 7.835GiB   4


![Рисунок 12 — Запуск через Docker Compose, статус healthy и журнал проверок](screenshots/term_compose.png)

*Рисунок 12 — Запуск через Docker Compose, статус healthy и журнал проверок*

![Рисунок 13 — Латентность, логи и ресурсы контейнера](screenshots/term_ops.png)

*Рисунок 13 — Латентность, логи и ресурсы контейнера*

# 11. Воспроизводимость

Зафиксированы: версии пакетов (`requirements.txt`, `requirements.lock.txt`, `requirements-api.txt`), базовый образ `python:3.12-slim`, снимок данных (CSV в проекте + SHA-256 в каждом run и в metadata.json), разбиение (`split.json` с индексами строк), `random_state = 42` для разбиения и случайного леса, параметры каждого run в MLflow, а для доставки — `exported_model/` с номером версии и эталонные `sample_request.json` / `expected_predictions.json`. Повтор шести команд `train.py` даёт те же метрики: повторный run «rf 200/12» при регистрации получил точно такие же val MAE = 1,0898 и RMSE = 1,8187, как исходный эксперимент.

# 12. Выводы

1. Для варианта 16 построен воспроизводимый цикл: данные → 8 run в MLflow (6 экспериментов + 2 регистрационных) → Model Registry → экспорт → API → Docker → откат → Compose.
2. Y1 исключена из признаков (корреляция с Y2 = 0,976 — явная утечка); исключение проверяется в коде обучения и в схеме API.
3. По validation MAE лучшая модель — RandomForestRegressor (200 деревьев, глубина 12): val MAE 1,090, test MAE 1,173, R² = 0,966. Ridge даёт MAE ≈ 2,0 и ограничен линейностью; усиление регуляризации его только ухудшает. Важна не только метрика: лес нелинеен и требует больше памяти, Ridge проще и интерпретируемее, но вдвое ошибается сильнее.
4. В Registry две версии, alias `champion` указывает на версию 1; смена alias не меняет работающий образ — нужен экспорт и новая сборка.
5. API валидирует вход: все 4 типа ошибочных запросов дают 422, ни одного 500; ответы API в контейнере совпадают с экспортированной моделью.
6. Обновление v1 → v2 и откат v2 → v1 выполнены без переобучения; откат — перезапуск сохранённого образа, версия подтверждена через `/metadata`.
7. Через Docker Compose сервис запускается одной командой и получает статус healthy.

# 13. Список источников

1. MLflow Documentation: Tracking, Model Registry, Aliases. https://mlflow.org/docs/latest/
2. Tsanas A., Xifara A. Energy Efficiency [Dataset]. UCI Machine Learning Repository, 2012. https://doi.org/10.24432/C51307
3. Tsanas A., Xifara A. Accurate quantitative estimation of energy performance of residential buildings using statistical machine learning tools // Energy and Buildings. 2012. Vol. 49. P. 560–567.
4. scikit-learn: Pipeline, ColumnTransformer, Ridge, RandomForestRegressor, Model evaluation. https://scikit-learn.org/stable/
5. FastAPI Documentation: Request Body, Lifespan Events. https://fastapi.tiangolo.com/
6. Docker Documentation: Dockerfile reference, Docker Compose, Healthcheck. https://docs.docker.com/
7. Методические указания к лабораторной работе 4 «MLOps». НГТУ, 2026.

# Приложение. Исходный код, зависимости, Dockerfile, Compose, примеры запросов

# Приложение. Исходный код

Все файлы проекта лежат в репозитории: `src/`, `Dockerfile`, `compose.yaml`, `requirements*.txt`. Ниже выведены основные.

In [56]:
!cat src/train.py

"""Обучение одной модели и запись run в MLflow.

Примеры:
  python src/train.py --algorithm ridge --alpha 1
  python src/train.py --algorithm rf --trees 200 --depth 12 --register
"""
import argparse
import tempfile
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
from mlflow.models import infer_signature
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import config as C


def build_pipeline(a):
    # preprocessing входит в Pipeline -> сохраняется вместе с моделью
    pre = ColumnTransformer([
        ("num", StandardScaler(), C.NUMERIC),
        ("cat", OneHotEncoder(handle_unknown="ignore"), C.CATEGORICAL),
    ])
    

In [57]:
!cat src/app.py

"""FastAPI-сервис прогноза нагрузки охлаждения (Y2). Модель загружается один раз при старте."""
import json
import logging
import os
import time
from contextlib import asynccontextmanager
from pathlib import Path

import mlflow.pyfunc
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, ConfigDict, Field

MODEL_DIR = Path(os.getenv("MODEL_DIR", Path(__file__).resolve().parents[1] / "exported_model"))
log = logging.getLogger("uvicorn.error")
state = {}


class Building(BaseModel):
    """Ровно 8 признаков геометрии здания; лишние поля запрещены, NaN/inf запрещены."""
    model_config = ConfigDict(extra="forbid")
    relative_compactness: float = Field(gt=0, allow_inf_nan=False)
    surface_area: float = Field(gt=0, allow_inf_nan=False)
    wall_area: float = Field(gt=0, allow_inf_nan=False)
    roof_area: float = Field(gt=0, allow_inf_nan=False)
    overall_height: float = Field(gt=0, allow_inf_nan=False)
    orientation: float = Field(ge=0, all

In [58]:
!cat Dockerfile

FROM python:3.12-slim

ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1 \
    MODEL_DIR=/app/exported_model

WORKDIR /app

COPY requirements-api.txt .
RUN python -m pip install --no-cache-dir -r requirements-api.txt

COPY src/app.py src/app.py
COPY exported_model exported_model

USER 10001:10001
EXPOSE 8000
CMD ["python", "-m", "uvicorn", "app:app", "--app-dir", "src", "--host", "0.0.0.0", "--port", "8000"]


In [59]:
!cat compose.yaml

services:
  api:
    build: .
    image: cooling-api:${IMAGE_TAG:-v1}
    container_name: cooling-api-compose
    ports:
      - "127.0.0.1:8000:8000"
    restart: unless-stopped
    healthcheck:
      test: ["CMD", "python", "-c", "import urllib.request,sys; sys.exit(0 if urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=3).status == 200 else 1)"]
      interval: 10s
      timeout: 5s
      retries: 3
      start_period: 20s


In [60]:
!cat requirements.txt

mlflow==3.5.0
scikit-learn==1.7.2
pandas==2.2.3
numpy==2.1.3
fastapi==0.118.0
uvicorn==0.37.0
pydantic==2.11.9
requests==2.32.5
sqlalchemy==2.0.43
